# Mall Customer Segmentation

**Unsupervised Learning | K-Means Clustering**

## Project Overview
This project segments mall customers using **Annual Income** and **Spending Score**. The goal is to identify groups of customers with similar purchasing behavior and summarize the characteristics of each segment.

### Workflow
1. Load and validate the dataset
2. Explore the data with descriptive statistics and visualizations
3. Select and scale clustering features
4. Explore the number of clusters using the Elbow Method and Silhouette Score
5. Train and evaluate the final K-Means model
6. Interpret and visualize customer segments

> **Note:** K-Means is an unsupervised algorithm. Cluster IDs are arbitrary labels, not ranked categories. Segment descriptions below are generated from the cluster centers.

## 1. Imports and Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_CLUSTERS = 5

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", None)

## 2. Load the Dataset

Place `Mall_Customers.csv` inside `data/raw/` before running this notebook.

In [ ]:
DATA_PATH = Path("../data/raw/Mall_Customers.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}\n"
        "Download Mall_Customers.csv and place it in data/raw/."
    )

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Initial Data Inspection

In [ ]:
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.info())
display(df.describe(include="all"))

In [ ]:
print("Missing values per column:")
display(df.isna().sum())

print(f"Duplicate rows: {df.duplicated().sum()}")

if df.isna().any().any():
    raise ValueError("The dataset contains missing values. Handle them before clustering.")

if df.duplicated().any():
    print("Warning: duplicate rows were found. Review them before proceeding.")

In [ ]:
required_columns = {
    "CustomerID",
    "Gender",
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)",
}

missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

display(df["Gender"].value_counts())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(data=df, x="Age", bins=15, kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.histplot(data=df, x="Annual Income (k$)", bins=15, kde=True, ax=axes[1])
axes[1].set_title("Annual Income Distribution")

sns.histplot(data=df, x="Spending Score (1-100)", bins=15, kde=True, ax=axes[2])
axes[2].set_title("Spending Score Distribution")

plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
display(df[numeric_columns].agg(["min", "max", "mean", "median", "std"]).T)

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    alpha=0.8,
)
plt.title("Annual Income vs. Spending Score")
plt.tight_layout()
plt.show()

## 5. Feature Selection

For this segmentation task, we use annual income and spending score. `CustomerID` is an identifier and is not a meaningful clustering feature.

In [ ]:
FEATURES = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[FEATURES].copy()

print(f"Features: {FEATURES}")
print(f"Feature matrix shape: {X.shape}")
display(X.head())

## 6. Feature Scaling

K-Means uses distance calculations, so features should be placed on a comparable scale. We fit the scaler on the selected features and use the scaled values for clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_summary = pd.DataFrame(
    X_scaled,
    columns=FEATURES,
).agg(["mean", "std"]).T

display(scaled_summary)

## 7. Choosing the Number of Clusters

We inspect both inertia (Elbow Method) and Silhouette Score. Neither metric should be used in isolation; interpret them alongside the cluster visualization and the business context.

In [ ]:
k_values = range(1, 11)
inertia_values = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
    )
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

In [ ]:
silhouette_k_values = range(2, 11)
silhouette_values = []

for k in silhouette_k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
    )
    labels = model.fit_predict(X_scaled)
    silhouette_values.append(silhouette_score(X_scaled, labels))

silhouette_results = pd.DataFrame({
    "n_clusters": list(silhouette_k_values),
    "silhouette_score": silhouette_values,
})

display(silhouette_results.sort_values("silhouette_score", ascending=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    silhouette_results["n_clusters"],
    silhouette_results["silhouette_score"],
    marker="o",
)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by Number of Clusters")
plt.xticks(list(silhouette_k_values))
plt.tight_layout()
plt.show()

## 8. Train the Final K-Means Model

The original analysis uses **5 clusters**. We retain that choice for this project so the segments can be examined consistently; the metrics above provide evidence to revisit this choice if needed.

In [ ]:
final_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10,
)

cluster_labels = final_model.fit_predict(X_scaled)
df["Cluster"] = cluster_labels

final_silhouette = silhouette_score(X_scaled, cluster_labels)
print(f"Silhouette Score (k={N_CLUSTERS}): {final_silhouette:.4f}")

display(df.head())
display(df["Cluster"].value_counts().sort_index().rename("customer_count"))

## 9. Visualize Customer Segments

In [ ]:
centers_original_scale = scaler.inverse_transform(final_model.cluster_centers_)
centers_df = pd.DataFrame(centers_original_scale, columns=FEATURES)
centers_df.index.name = "Cluster"

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x=FEATURES[0],
    y=FEATURES[1],
    hue="Cluster",
    palette="viridis",
    s=65,
    alpha=0.8,
)

plt.scatter(
    centers_df[FEATURES[0]],
    centers_df[FEATURES[1]],
    marker="X",
    s=250,
    color="red",
    edgecolor="black",
    label="Centroids",
)

plt.title("Mall Customer Segmentation with K-Means")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 10. Cluster Analysis and Interpretation

In [ ]:
cluster_summary = df.groupby("Cluster").agg(
    customer_count=("CustomerID", "count"),
    average_age=("Age", "mean"),
    average_income=("Annual Income (k$)", "mean"),
    average_spending_score=("Spending Score (1-100)", "mean"),
).round(2)

display(cluster_summary)

In [ ]:
# Create descriptive labels from each cluster center.
# The labels describe relative income/spending levels in this dataset.
income_midpoint = df["Annual Income (k$)"].median()
spending_midpoint = df["Spending Score (1-100)"].median()

def describe_segment(row):
    income_level = "High Income" if row["Annual Income (k$)"] >= income_midpoint else "Low Income"
    spending_level = (
        "High Spending"
        if row["Spending Score (1-100)"] >= spending_midpoint
        else "Low Spending"
    )
    return f"{income_level} / {spending_level}"

segment_names = {
    cluster_id: describe_segment(center)
    for cluster_id, center in centers_df.iterrows()
}

df["Customer Segment"] = df["Cluster"].map(segment_names)

cluster_summary["segment"] = cluster_summary.index.map(segment_names)
display(cluster_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cluster_summary["average_income"].plot(kind="bar", ax=axes[0])
axes[0].set_title("Average Annual Income")
axes[0].set_ylabel("Income (k$)")
axes[0].set_xlabel("Cluster")

cluster_summary["average_spending_score"].plot(kind="bar", ax=axes[1])
axes[1].set_title("Average Spending Score")
axes[1].set_ylabel("Spending Score")
axes[1].set_xlabel("Cluster")

cluster_summary["customer_count"].plot(kind="bar", ax=axes[2])
axes[2].set_title("Number of Customers")
axes[2].set_ylabel("Customers")
axes[2].set_xlabel("Cluster")

for ax in axes:
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 11. Export Results

The original dataset remains unchanged. The enriched customer-level output and cluster summary are saved to `reports/`.

In [ ]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(REPORTS_DIR / "customer_segments.csv", index=False)
cluster_summary.to_csv(REPORTS_DIR / "cluster_summary.csv")

print("Reports saved to:", REPORTS_DIR.resolve())

## 12. Key Takeaways

- K-Means grouped customers using annual income and spending score.
- The Elbow Method and Silhouette Score were used to explore the number of clusters.
- Cluster centers and summary statistics help describe the resulting segments.
- Segment labels are relative descriptions based on this dataset, not universal customer categories.

### Limitations and Next Steps
- Results depend on the selected features, scaling method, and number of clusters.
- K-Means favors compact, roughly spherical clusters and can be sensitive to outliers.
- Further work could compare alternative clustering algorithms, test feature combinations, and validate whether the segments support useful marketing decisions.